In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import xarray as xr
import numpy as np
from distributed import Client, LocalCluster
import dask
import pickle
import os
from scipy.stats import linregress
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import linregress
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
import joblib # For saving our models
import time
import emcee
from matplotlib.ticker import FuncFormatter
plt.rcParams['text.usetex'] = True
import corner # The library for creating corner plots

# Loadings need for LaTex

In [2]:
# Get the current PATH
original_path = os.environ.get('PATH', '')
latex_path = '/glade/u/apps/casper/24.12/spack/opt/spack/texlive/20240312/gcc/12.4.0/nm7e/bin/x86_64-linux' 
# Prepend the LaTeX path to the system's PATH
# We prepend it to make sure it's found first.
os.environ['PATH'] = latex_path + os.pathsep + original_path
plt.rcParams['text.usetex'] = True

# Read in pickle files

In [3]:
#======================
# Load CNTL Dictionaryp
#======================
read_path = '/glade/u/home/mckenna/scratch/ppe_processed_files/'
dict_list = sorted(glob.glob(read_path + '*.pkl'))

cntl_path_id = [i for i, f in enumerate(dict_list) if '_00' in f]
cntl_dict_path = dict_list[cntl_path_id[0]]

with open(cntl_dict_path, 'rb') as f:
    cntl_dict = pickle.load(f)

cntl_pd_dict = cntl_dict['pD']

In [49]:
#dict_list

In [4]:
def compute_warm_stratiform_mask(case_dict):
    # --- 1. Define Instantaneous Masks ---
    iwp = case_dict['iwp']
    icc = case_dict['icc']
    ttop = case_dict['ttop']
    lcc = case_dict['lcc']
    ocnfrac = case_dict['ocnfrac']
    omega_500 = case_dict['omega_500']
    omega_700 = case_dict['omega_700']
    th7001000 = case_dict['th7001000']
    
    warm_mask = ((iwp < 1e-3) & (icc < 1e-12) & (ttop > 273.15) & (lcc > 0.001) & (ocnfrac > 0.99))
    warm_overcast_mask = (warm_mask & (lcc > 0.9) & (ocnfrac > 0.99))
    strat_mask_ms = ((omega_500 > 10.) & (omega_700 > 10.) & (th7001000 > 18.55) & (ocnfrac > 0.99))

    # --- 2. Calculate Climatological Fractions (for visualization and regime definition) ---
    warm_frac = warm_mask.mean(axis=0)
    warm_overcast_frac = warm_overcast_mask.mean(axis=0)
    strat_frac = strat_mask_ms.mean(axis=0)

    # --- 3. Define the Final 'warm_strat' Regime Mask (for analysis) ---
    strat_column_mask = strat_frac > 0.3
    strat_column_mask_broadcast = np.broadcast_to(strat_column_mask, warm_mask.shape)
    warm_strat_mask = warm_mask & strat_column_mask_broadcast

    # --- 4. Calculate the 'warm_strat' Fraction (for visualization) ---
    warm_strat_frac = np.where(strat_column_mask, warm_frac, np.nan)

    # --- 5. Create Masked Arrays for Plotting ---
    # Using a small positive threshold like 0.01 is often safer than 0.0
    plot_thresh = 0.01
    warm_masked = np.ma.masked_where(warm_frac < plot_thresh, warm_frac)
    warm_overcast_masked = np.ma.masked_where(warm_overcast_frac < plot_thresh, warm_overcast_frac)
    strat_masked = np.ma.masked_where(strat_frac < plot_thresh, strat_frac)
    warm_strat_masked = np.ma.masked_where(np.isnan(warm_strat_frac) | (warm_strat_frac < plot_thresh), warm_strat_frac)

    # --- 6. Return all necessary products ---
    return {
        # The crucial analysis masks
        'warm_mask': warm_mask,
        'warm_overcast_mask': warm_overcast_mask,
        'strat_mask': strat_mask_ms,
        'warm_strat_mask': warm_strat_mask, # The primary output for compute_metrics
        
        # The fractions for visualization
        'warm_frac': warm_frac,
        'warm_overcast_frac': warm_overcast_frac,
        'strat_frac': strat_frac,
        'warm_strat_frac': warm_strat_frac,
        
        # The plottable masked arrays
        'warm_masked': warm_masked,
        'warm_overcast_masked': warm_overcast_masked,
        'strat_masked': strat_masked,
        'warm_strat_masked': warm_strat_masked,
    }

In [136]:
results = {}

dumi=0
for path in dict_list:
    case_name = os.path.basename(path).replace('.pkl', '')
    print('case_name:',case_name)


    print(f'Processing: {case_name}','; % done:',(dumi+1)/len(dict_list)*100.)
    with open(path, 'rb') as f:
        case = pickle.load(f)

    auto_fac = case['auto_fac']
    accr_fac = case['accr_fac']
    print('auto_fac :',auto_fac)
    print('accr_fac :',accr_fac)
    pd_dict = case['pD']
    pi_dict = case['pI']


    #========================================
    # This block computes masks for:
    # (1) Global (averaged w/o clouds)
    # (2) Warm clouds
    # (3) Warm marine overcast clouds
    # (4) Warm marine overcast stratiform clouds, 
    # following Medeiros & Stevens (2011)
    #========================================
    pd_mask_dict = compute_warm_stratiform_mask(pd_dict)
    pi_mask_dict = compute_warm_stratiform_mask(pi_dict)
    cntl_pd_mask_dict = compute_warm_stratiform_mask(cntl_pd_dict)
    
    for key in pd_mask_dict.keys():
        pd_dict[key] = pd_mask_dict[key]
        pi_dict[key] = pi_mask_dict[key]
        cntl_pd_dict[key] = cntl_pd_mask_dict[key]

    plot_regime(pd_dict,'PD',auto_fac,accr_fac,case_name)
    plot_regime(pi_dict,'PI',auto_fac,accr_fac,case_name)
    
    #print(aaaaaaa)

    dumi+=1


case_name: msp1_00
Processing: msp1_00 ; % done: 1.8518518518518516
auto_fac : 1.0
accr_fac : 1.0
cloud_occurrence_frequency_00_PD.png
cloud_occurrence_frequency_00_PI.png
case_name: msp1_01
Processing: msp1_01 ; % done: 3.7037037037037033
auto_fac : 299.0926451671614
accr_fac : 22.47762310145409
cloud_occurrence_frequency_01_PD.png
cloud_occurrence_frequency_01_PI.png
case_name: msp1_02
Processing: msp1_02 ; % done: 5.555555555555555
auto_fac : 1.0875281829072527
accr_fac : 5.489066554702007
cloud_occurrence_frequency_02_PD.png
cloud_occurrence_frequency_02_PI.png
case_name: msp1_03
Processing: msp1_03 ; % done: 7.4074074074074066
auto_fac : 112.1861726522124
accr_fac : 2.953302346185345
cloud_occurrence_frequency_03_PD.png
cloud_occurrence_frequency_03_PI.png
case_name: msp1_04
Processing: msp1_04 ; % done: 9.25925925925926
auto_fac : 0.0023806549990427
accr_fac : 0.1534184737437596
cloud_occurrence_frequency_04_PD.png
cloud_occurrence_frequency_04_PI.png
case_name: msp1_05
Processin

In [135]:
def plot_regime(in_dict,pd_or_pi,auto_fac,accr_fac,case_name):
    fig = plt.figure(figsize=(12, 7), constrained_layout=True)
    Fontsize = 16
    titles = [
        'Warm Cloud Occurrence Frequency',
        'Warm Overcast Cloud Occurrence Frequency',
        'Sc Conditions Occurrence Frequency\n[Medeiros and Stevens (2011)]',
        'Warm Sc Cloud Occurrence Frequency'
    ]
    
    cbar_labels = [r'$cf_{\mathrm{warm}}$', r'$cf_{\mathrm{warm,overcast}}$',r'$f_{\mathrm{Sc}}$', r'$cf_{\mathrm{warm}} \,|\, f_{\mathrm{Sc}} > 0.3$']
    vars_to_plot = ['warm_masked', 'warm_overcast_masked', 'strat_masked', 'warm_strat_masked']
    
    # Common color settings
    #bounds = np.arange(0.1, 0.6 + 0.1, 0.1)
    #n_levels = len(bounds) - 1
    #cmap = plt.get_cmap('RdPu_r', n_levels)
    #norm = mpl.colors.BoundaryNorm(boundaries=bounds, ncolors=cmap.N)
    
    labs = ['(a)','(b)','(c)','(d)']
    for i, var in enumerate(vars_to_plot):
        ax = fig.add_subplot(2, 2, i + 1, projection=ccrs.PlateCarree())
        ax.set_global()
        ax.coastlines()
        ax.tick_params(labelsize=Fontsize)

        if var == 'strat_masked':
            # Define your plotting threshold here. This is independent of any previous masking.
            plot_display_threshold = 0.1 
            bounds = np.arange(0.1, 0.6 + 0.1, 0.1)
            n_levels = len(bounds) - 1
            cmap = plt.get_cmap('RdPu_r', n_levels)
            norm = mpl.colors.BoundaryNorm(boundaries=bounds, ncolors=cmap.N)
        else:
            # Define your plotting threshold here. This is independent of any previous masking.
            plot_display_threshold = 0.01
            #plot_display_threshold = 0.1
            bounds = np.concatenate((np.array([plot_display_threshold]),np.arange(0.1, 0.6 + 0.1, 0.1)))
            #bounds = np.arange(0.1, 0.6 + 0.1, 0.1)
            n_levels = len(bounds) - 1
            cmap = plt.get_cmap('RdPu_r', n_levels)
            norm = mpl.colors.BoundaryNorm(boundaries=bounds, ncolors=cmap.N)

        # --- APPLY MASKING AT PLOT TIME ---
        # Get the raw fraction data
        data_to_plot = in_dict[var]
        # Create a masked version for this specific plot
        plot_data = np.where(data_to_plot >= plot_display_threshold, data_to_plot, np.nan)
        # ---------------------------------
        
        sc = ax.scatter(
            in_dict['lon'], in_dict['lat'],
            c=plot_data,
            cmap=cmap, norm=norm, s=10
        )
        cbar = fig.colorbar(sc, ax=ax, pad=0.01, shrink=0.7, ticks=bounds, boundaries=bounds)
        cbar.ax.tick_params(labelsize=Fontsize)
        cbar.set_label(cbar_labels[i], fontsize=Fontsize)
        ax.set_title(titles[i], fontsize=Fontsize)
        ax.text(0.01,0.99,labs[i],transform=ax.transAxes,fontsize=Fontsize*1.5,ha='left',va='top')
    if pd_or_pi == 'PD':   
        plt.suptitle(f'Present Day\nx_auto={str(np.around(auto_fac,4))}; x_accr={str(np.around(accr_fac,4))}', fontsize=Fontsize*2.,x=0.5,ha='center')
    else:
        plt.suptitle(f'Pre-Industrial\nx_auto={str(np.around(auto_fac,4))}; x_accr={str(np.around(accr_fac,4))}', fontsize=Fontsize*2.,x=0.5,ha='center')


    save_path = '/glade/u/home/mckenna/scratch/figures/ppe_cloud_masks/'
    dum_str = case_name.split('_')[-1]
    file_name = f'cloud_occurrence_frequency_{dum_str}_{pd_or_pi}.png'
    print(file_name)
    #plt.savefig(save_path+file_name,dpi=300)
    #plt.show()
    plt.close()

    return

In [ ]:
plot_regime(pd_dict,'PD',auto_fac,accr_fac,case_name)

In [ ]:
plot_regime(pi_dict,'PI',auto_fac,accr_fac,case_name)

# Map of (temporal) cloud fraction for warm, overcast, marine stratocumlus defined using Medeiros and Stevens (2011)

In [10]:
cntl_pd_mask_dict = compute_warm_stratiform_mask(cntl_pd_dict)

for key in cntl_pd_mask_dict.keys():
    cntl_pd_dict[key] = cntl_pd_mask_dict[key]

In [54]:
def plot_regime_cntl_SI_fig(in_dict):
    fig = plt.figure(figsize=(8,6), constrained_layout=True)
    Fontsize = 16

    ax = fig.add_subplot(111, projection=ccrs.PlateCarree())
    ax.set_global()
    ax.coastlines()
    ax.tick_params(labelsize=Fontsize)

    # Define your plotting threshold here. This is independent of any previous masking.
    plot_display_threshold = 0.1 
    bounds = np.arange(0.1, 0.6 + 0.1, 0.1)
    n_levels = len(bounds) - 1
    cmap = plt.get_cmap('RdPu_r', n_levels)
    norm = mpl.colors.BoundaryNorm(boundaries=bounds, ncolors=cmap.N)

    data_to_plot = in_dict['strat_masked']
    plot_data = np.where(data_to_plot >= plot_display_threshold, data_to_plot, np.nan)
    
    sc = ax.scatter(
        in_dict['lon'], in_dict['lat'],
        c=plot_data,
        cmap=cmap, norm=norm, s=3
    )
    cbar = fig.colorbar(sc, ax=ax, pad=0.01, shrink=0.55, ticks=bounds, boundaries=bounds)
    cbar.ax.tick_params(labelsize=Fontsize)
    cbar.set_label(r'$f_{\mathrm{Sc}}$', fontsize=Fontsize)


    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linewidth=1, color='gray', alpha=0.5, linestyle='--')
    
    gl.top_labels = False
    gl.right_labels = False
    
    # Add axis titles (optional, but good practice)
    ax.set_xlabel('Longitude', fontsize=Fontsize)
    ax.set_ylabel('Latitude', fontsize=Fontsize)

    plt.suptitle(r'\textbf{Marine Sc Cloud Occurrence Fraction following Medeiros and Stevens (2011)}',fontsize=Fontsize*0.9,y=0.825)
    save_path = '/glade/u/home/mckenna/work/figures/pnas_paper/'
    file_name = f'cloud_occurrence_frequency_CNTL_PD.png'
    print(file_name)
    plt.savefig(save_path+file_name,dpi=300)
    #plt.show()
    plt.close()

    return

In [55]:
plot_regime_cntl_SI_fig(cntl_pd_dict)

cloud_occurrence_frequency_CNTL_PD.png
